# EXP-033 · NaFlex spatial representation on Kaggle

Use a GPU T4 accelerator and attach the private data and runtime ZIPs. Training starts disabled. Use the configuration below to run one authorized stage at a time.

1. **Preflight** measures pretrained CUDA memory, native/cache parity, FP16 throughput, and optimizer replay. Save the verified export, upload it as a private Kaggle dataset, then open a fresh session and attach that dataset.
2. **Verify publication** reads the checkpoint from the attached archive. This confirms that the exported canary can be restored from a separate Kaggle session.
3. **Five-fold CV** is a separate owner decision. Set the explicit authorization, a time budget with room left for export, and requested folds. Resume with the latest previously published CV ZIP if needed. Outer scoring waits for all five sealed folds.


In [ ]:
STAGE = "preflight"  # validate, preflight, verify-publication, or cv
AUTHORIZE_STAGE = False
AUTHORIZE_FULL_CV = False

INPUT_ZIP = "/kaggle/input/exp033-p2-private/exp033_inputs.zip"
RUNTIME_ZIP = "/kaggle/input/exp033-runtime-private/exp033_runtime.zip"
PUBLISHED_PREFLIGHT_ZIP = ""  # from a previous saved session, attached as private data
PUBLISHED_PREFLIGHT_SHA256 = ""
RESUME_CV_ZIP = ""  # optional, attach the last published cv export
RESUME_CV_SHA256 = ""
MODEL_DIRECTORY = ""  # optional verified local model assets; otherwise download pinned files
FOLDS = [1, 2, 3, 4, 5]
MAX_SECONDS = 1800
DEVICE = "cuda:0"

SOURCE_COMMIT = "d54ad4d7b030aaeaf768088eff93292ae7cd4793"
RUNTIME_SHA = "1ba0f6c3a7d7be4bc49a65f16022e24377ed0047a5b2fad53a4053dbeb9c9793"
INPUT_SHA = "3bfb284878c886c7313875f909171d3d015ee7860790375a7f873e904b106d84"
MANIFEST_SHA = "21f47c3d3c44530b2ea55d12368b46bf1050d665de607094bce591283baf1a82"


In [ ]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import subprocess
import sys
import zipfile


def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def mounted(value):
    path = Path(value).resolve()
    if not value or not path.is_relative_to("/kaggle/input") or not path.is_file():
        raise ValueError(f"Attach and locate the private input under /kaggle/input: {value or '(unset)'}")
    return path


def run():
    if not AUTHORIZE_STAGE:
        print("Execution is disabled. Review the configuration and authorize the chosen stage.")
        return
    if STAGE not in {"validate", "preflight", "verify-publication", "cv"}:
        raise ValueError("Unknown stage")
    if STAGE == "cv" and not AUTHORIZE_FULL_CV:
        raise ValueError("CV needs its own explicit authorization")
    if not 0 < MAX_SECONDS <= 36_000 or not FOLDS or not set(FOLDS) <= set(range(1, 6)):
        raise ValueError("Set a finite stage budget and valid fold IDs")
    runtime, inputs_zip = mounted(RUNTIME_ZIP), mounted(INPUT_ZIP)
    if sha(runtime) != RUNTIME_SHA or sha(inputs_zip) != INPUT_SHA:
        raise ValueError("Runtime or data ZIP checksum changed")
    attempt = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    local = Path("/kaggle/temp") / f"exp033_{attempt}"
    repo, output = local / "source", local / "run"
    repo.mkdir(parents=True)
    # RUNTIME_SHA pins the exact git archive contents.
    with zipfile.ZipFile(runtime) as bundle:
        names = bundle.namelist()
        for name in names:
            item = PurePosixPath(name)
            if item.is_absolute() or ".." in item.parts or "\\" in name:
                raise ValueError(f"Invalid runtime ZIP member: {name}")
        bundle.extractall(repo)
    helper = repo / "scripts/exp033_artifacts.py"
    if STAGE == "cv" and RESUME_CV_ZIP:
        subprocess.run([sys.executable, str(helper), "restore", str(mounted(RESUME_CV_ZIP)),
                        str(output), "--sha", RESUME_CV_SHA256], check=True)
    output.mkdir(exist_ok=True)
    logs = output / f"session_{attempt}.log"
    env = os.environ.copy()
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    env["HF_HOME"] = str(Path("/kaggle/temp") / "exp033_model_cache")
    environment = local / "venv"
    exe = environment / "bin/python"
    artifact = Path("/kaggle/working") / f"exp033_{STAGE}_{attempt}.zip"

    def execute(cmd, log):
        process = subprocess.Popen([str(p) for p in cmd], cwd=repo, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="")
            if process.wait() != 0:
                raise RuntimeError(f"Worker exited {process.returncode}, see {logs.name}")
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait(timeout=30)

    try:
        with logs.open("w", encoding="utf-8") as log:
            print(f"EXP-033 {STAGE} at worker {SOURCE_COMMIT}", file=log)
            if STAGE != "validate":
                execute(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
                         "--format=csv"], log)
            bootstrap = local / "uv_pkg"
            execute([sys.executable, "-m", "pip", "install", "--target", bootstrap, "uv==0.8.15"], log)
            env["PYTHONPATH"] = str(bootstrap)
            env["UV_PYTHON_INSTALL_DIR"] = str(Path("/kaggle/temp") / "exp033_python")
            uv = [sys.executable, "-m", "uv"]
            execute(uv + ["python", "install", "3.12.10"], log)
            execute(uv + ["venv", "--python", "3.12.10", environment], log)
            cuda = STAGE != "validate"
            execute(uv + ["pip", "install", "--python", exe, "torch==2.7.1+cu126" if cuda else "torch==2.7.1",
                          "--index-url", "https://download.pytorch.org/whl/" + ("cu126" if cuda else "cpu")], log)
            execute(uv + ["pip", "install", "--python", exe, "-r", repo / "requirements-exp033.lock.txt"], log)
            execute(uv + ["pip", "check", "--python", exe], log)
            execute(uv + ["pip", "freeze", "--python", exe], log)
            env.pop("PYTHONPATH")
            cmd = [exe, "-u", repo / "scripts/run_exp033.py", STAGE, "--input-zip", inputs_zip,
                   "--inputs", Path("/kaggle/temp") / f"exp033_inputs_{MANIFEST_SHA[:16]}",
                   "--output", output, "--zip-sha", INPUT_SHA, "--manifest-sha", MANIFEST_SHA,
                   "--device", DEVICE, "--max-seconds", MAX_SECONDS, "--folds", *FOLDS]
            if MODEL_DIRECTORY:
                cmd += ["--model-dir", Path(MODEL_DIRECTORY)]
            if STAGE in ("verify-publication", "cv"):
                published = local / "published"
                execute([sys.executable, helper, "restore", mounted(PUBLISHED_PREFLIGHT_ZIP), published,
                         "--sha", PUBLISHED_PREFLIGHT_SHA256], log)
                cmd += ["--published", published, "--published-sha", sha(published / "preflight.json")]
            if STAGE == "cv":
                cmd += ["--authorize-cv"]
            execute(cmd, log)
            for name in ("preflight.json", "decision.json", "publication_verified.json"):
                receipt = output / name
                if receipt.is_file():
                    info = json.loads(receipt.read_text(encoding="utf-8"))
                    print(name, json.dumps({k: v for k, v in info.items() if k not in ("identity", "metrics", "paired")}, indent=2))
    finally:
        subprocess.run([sys.executable, str(helper), "export", str(output), str(artifact)], check=True)
        print("Export verified:", artifact.name)
        print("Archive SHA-256:", sha(artifact))
        print("Publish or download this archive before stopping the runtime. Local session storage is temporary.")


run()


### Resume and interpret

In a fresh session, attach the previous preflight archive and the latest completed CV archive as private datasets. Check both SHA-256 values before restoring. A failed or timed-out run exports completed epochs and logs. After an unexpected termination, recover from the last archive already published outside the runtime. Use a single writer per run. The frozen bank may be rebuilt from its pins. Results report the independent and fixed-fusion comparisons separately. All scores are exploratory P2 evidence.
